# Générateur de données — Formation Causalité

Notebook de génération de données simulées illustrant quatre biais causaux classiques.
Tous les paramètres de simulation se trouvent dans la cellule **PARAMÈTRES** ci-dessous.

**Usage :** Kernel → Restart & Run All — aucune modification nécessaire hors de la cellule Paramètres.

---
Scénarios couverts :
- **Scénario 0** — Biais de petits nombres (variance binomiale)
- **Scénarios 1a/1b/1c** — Biais de sélection (confondant)
- **Scénario 2** — Surcontrôle sur un médiateur
- **Scénario 3** — Surcontrôle sur un collider

In [ ]:
# ═══════════════════════════════════════════════════════════
# PARAMÈTRES — modifiez ici uniquement
# ═══════════════════════════════════════════════════════════

SEED        = 42
N_MAGASINS  = 200
T_MOIS      = 24

N_PETIT  = 30
N_MOYEN  = 150
N_GRAND  = 500

P_BASE_VISITE  = 0.05
EFFET_URBAIN   = 0.03
EFFET_EQUIPE   = 0.02
EFFET_SAISON   = {1: -0.01, 2: -0.01, 3: 0.0, 4: 0.01, 5: 0.02, 6: 0.02,
                   7: 0.02, 8: 0.01, 9: 0.0, 10: -0.01, 11: 0.01, 12: 0.02}

EFFET_PUB_VISITES = 0.10
EFFET_PUB_PANIER  = 0.10

MU_PANIER_BASE = 50.0
SIGMA_PANIER   = 15.0

P_PUB_BONNE_EQUIPE    = 0.70
P_PUB_MAUVAISE_EQUIPE = 0.30
P_PUB_URBAIN          = 0.65
P_PUB_RURAL           = 0.25
P_PUB_HAUTE_SAISON    = 0.70
P_PUB_BASSE_SAISON    = 0.30

PARAMS = {
    'seed': SEED,
    'n_magasins': N_MAGASINS,
    't_mois': T_MOIS,
    'n_petit': N_PETIT,
    'n_moyen': N_MOYEN,
    'n_grand': N_GRAND,
    'p_base_visite': P_BASE_VISITE,
    'effet_urbain': EFFET_URBAIN,
    'effet_equipe': EFFET_EQUIPE,
    'effet_saison': EFFET_SAISON,
    'effet_pub_visites': EFFET_PUB_VISITES,
    'effet_pub_panier': EFFET_PUB_PANIER,
    'mu_panier_base': MU_PANIER_BASE,
    'sigma_panier': SIGMA_PANIER,
    'p_pub_bonne_equipe': P_PUB_BONNE_EQUIPE,
    'p_pub_mauvaise_equipe': P_PUB_MAUVAISE_EQUIPE,
    'p_pub_urbain': P_PUB_URBAIN,
    'p_pub_rural': P_PUB_RURAL,
    'p_pub_haute_saison': P_PUB_HAUTE_SAISON,
    'p_pub_basse_saison': P_PUB_BASSE_SAISON,
}

In [ ]:
# ═══════════════════════════════════════════════════════════
# IMPORTS ET CONFIGURATION
# ═══════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import networkx as nx
import statsmodels.formula.api as smf
import seaborn as sns
from pathlib import Path

# Configuration matplotlib
matplotlib.rcParams['figure.dpi'] = 100
matplotlib.rcParams['font.size'] = 11

# RNG reproductible — seule source de hasard du notebook
# Passer rng en argument à chaque fonction qui en a besoin
# Ne jamais appeler np.random directement pour les tirages
rng = np.random.default_rng(SEED)

# Création automatique des dossiers d'export
Path('figures').mkdir(parents=True, exist_ok=True)
Path('data').mkdir(parents=True, exist_ok=True)

print(f"RNG initialisé avec SEED={SEED}")
print("Dossiers figures/ et data/ prets.")

---
## Générateur de panel partagé

Fonctions utilisées par tous les scénarios. `generate_base_panel` produit les caractéristiques structurelles. `compute_outcomes` calcule les variables d'outcome pour un DataFrame avec `pub` assigné.

In [ ]:
def generate_base_panel(params, rng):
    """
    Génère le panel partagé N_magasins × T_mois.
    Retourne : DataFrame avec colonnes :
        magasin_id, taille, n_potentiel, urbain, qualite_equipe, mois, effet_saison_val
    NB : pas de colonnes pub, nb_visites, ventes, panier_moyen — assignées par chaque scénario.
    """
    n = params['n_magasins']
    t = params['t_mois']
    n_potentiel_map = {
        'petit': params['n_petit'],
        'moyen': params['n_moyen'],
        'grand': params['n_grand'],
    }
    # Caractéristiques fixes (vectorisé)
    magasins = pd.DataFrame({
        'magasin_id': range(n),
        'taille': rng.choice(['petit', 'moyen', 'grand'], size=n, p=[0.4, 0.4, 0.2]),
        'urbain': rng.integers(0, 2, size=n),   # binaire 0/1
        'qualite_equipe': rng.integers(0, 2, size=n),
    })
    magasins['n_potentiel'] = magasins['taille'].map(n_potentiel_map)
    # Panel magasin × mois (cross join pandas 2.x)
    mois_df = pd.DataFrame({'mois': range(1, t + 1)})
    base = magasins.merge(mois_df, how='cross')
    base['effet_saison_val'] = base['mois'].map(params['effet_saison'])
    return base.reset_index(drop=True)


def compute_outcomes(df, params, rng):
    """
    Calcule p_visite, nb_visites, ventes, panier_moyen.
    Entrée : df avec colonnes taille, urbain, qualite_equipe, effet_saison_val, pub.
    p_visite est clippée à [0.01, 0.99] — OBLIGATOIRE avant rng.binomial().
    Approximation CLT pour ventes : ventes ~ Normal(nb_visites * mu, sqrt(nb_visites) * sigma).
    """
    p_visite = (
        params['p_base_visite']
        + params['effet_urbain'] * df['urbain']
        + params['effet_equipe'] * df['qualite_equipe']
        + df['effet_saison_val']
        + params['effet_pub_visites'] * df['pub']
    )
    p_visite = np.clip(p_visite, 0.01, 0.99)   # OBLIGATOIRE
    df = df.copy()
    df['p_visite'] = p_visite
    df['nb_visites'] = rng.binomial(df['n_potentiel'].values, p_visite.values)
    mu_panier = params['mu_panier_base'] * (1 + params['effet_pub_panier'] * df['pub'])
    df['ventes'] = (
        df['nb_visites'] * mu_panier
        + rng.normal(0, params['sigma_panier'], len(df)) * np.sqrt(df['nb_visites'])
    )
    df['panier_moyen'] = np.where(
        df['nb_visites'] > 0,
        df['ventes'] / df['nb_visites'],
        np.nan
    )
    return df

In [ ]:
# ─────────────────────────────────────────────
# Génération du panel de base (pub=0 partout)
# ─────────────────────────────────────────────
base_df = generate_base_panel(PARAMS, rng)

# Ajouter pub=0 pour calculer les outcomes de référence
base_df['pub'] = 0
base_df = compute_outcomes(base_df, PARAMS, rng)
# Retirer pub du panel de base après calcul des outcomes (chaque scénario réassigne)
base_df = base_df.drop(columns=['pub'])

print(f"Panel de base : {len(base_df)} lignes × {len(base_df.columns)} colonnes")
print(f"Colonnes : {list(base_df.columns)}")
print(base_df.head(3))

# Export CSV
base_df.to_csv('data/base_panel.csv', index=False)
print("\nExporte : data/base_panel.csv")